# Model Evaluation

This notebook demonstrates how to evaluate the trained AI model on various benchmarks. It covers:

1. Loading the trained model
2. Evaluating on code generation benchmarks
3. Evaluating on dialogue and QA benchmarks
4. Evaluating on reasoning benchmarks
5. Analyzing model performance
6. Generating evaluation reports

## Setup

First, let's install the required packages and import the necessary modules.

In [ ]:
# Install required packages
!pip install transformers datasets torch nltk rouge-score pyyaml tqdm matplotlib seaborn pandas

In [ ]:
# Import modules
import os
import sys
import logging
import yaml
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Add src directory to path
sys.path.append(os.path.abspath('../'))

# Import custom modules
from src.utils.metrics import (
    compute_bleu, compute_rouge, compute_meteor, compute_exact_match,
    compute_f1, compute_pass_at_k
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Load Configuration

Load the configuration file that defines the evaluation benchmarks.

In [ ]:
# Load configuration
with open("../configs/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Display evaluation configuration
print(f"Evaluation benchmarks:")
for benchmark in config['evaluation']['benchmarks']:
    print(f"  {benchmark['name']} ({benchmark['task']}): {benchmark['metric']}")

## Load Model

Load the trained model for evaluation.

In [ ]:
# Define model path
model_path = "../outputs/final_model"

# Check if model exists
if not os.path.exists(model_path):
    print(f"Model not found at {model_path}. Please run the model training notebook first.")
    # For demonstration, load a pretrained model
    model_path = "gpt2"
    print(f"Loading pretrained model {model_path} for demonstration.")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Loaded model from {model_path}")
print(f"Model device: {device}")

## Define Evaluation Functions

Define functions to evaluate the model on different benchmarks.

In [ ]:
def generate_text(prompt, max_length=100, temperature=0.7, top_p=0.9, top_k=50, num_return_sequences=1):
    """Generate text from a prompt."""
    # Tokenize prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    # Generate text
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode output
    generated_texts = [tokenizer.decode(out, skip_special_tokens=True) for out in output]
    
    return generated_texts

def evaluate_code_generation(dataset, num_samples=10, k_values=[1, 5, 10]):
    """Evaluate model on code generation tasks."""
    results = {}
    
    # Limit dataset size for demonstration
    if len(dataset) > num_samples:
        dataset = dataset.select(range(num_samples))
    
    # Generate code for each problem
    all_predictions = []
    all_references = []
    
    for example in tqdm(dataset, desc="Evaluating code generation"):
        # Get prompt and reference
        prompt = example.get('prompt', example.get('input', example.get('question', "")))
        reference = example.get('completion', example.get('output', example.get('answer', "")))
        
        # Generate multiple solutions
        predictions = generate_text(
            prompt,
            max_length=200,
            temperature=0.8,
            num_return_sequences=max(k_values)
        )
        
        # Remove prompt from predictions
        predictions = [pred.replace(prompt, "").strip() for pred in predictions]
        
        # Add to lists
        all_predictions.append(predictions)
        all_references.append(reference)
    
    # Compute pass@k for different k values
    for k in k_values:
        pass_at_k = compute_pass_at_k(all_predictions, all_references, k=k)
        results[f"pass@{k}"] = pass_at_k
    
    return results

def evaluate_dialogue(dataset, num_samples=10):
    """Evaluate model on dialogue tasks."""
    results = {}
    
    # Limit dataset size for demonstration
    if len(dataset) > num_samples:
        dataset = dataset.select(range(num_samples))
    
    # Generate responses for each prompt
    all_predictions = []
    all_references = []
    
    for example in tqdm(dataset, desc="Evaluating dialogue"):
        # Get prompt and reference
        prompt = example.get('prompt', example.get('input', example.get('question', "")))
        reference = example.get('completion', example.get('output', example.get('answer', "")))
        
        # Generate response
        prediction = generate_text(
            prompt,
            max_length=150,
            temperature=0.7
        )[0]
        
        # Remove prompt from prediction
        prediction = prediction.replace(prompt, "").strip()
        
        # Add to lists
        all_predictions.append(prediction)
        all_references.append(reference)
    
    # Compute BLEU scores
    bleu_scores = compute_bleu(all_predictions, [[ref] for ref in all_references])
    results.update(bleu_scores)
    
    # Compute ROUGE scores
    rouge_scores = compute_rouge(all_predictions, all_references)
    results.update(rouge_scores)
    
    # Compute METEOR score
    meteor_score = compute_meteor(all_predictions, all_references)
    results["meteor"] = meteor_score
    
    return results

def evaluate_qa(dataset, num_samples=10):
    """Evaluate model on question answering tasks."""
    results = {}
    
    # Limit dataset size for demonstration
    if len(dataset) > num_samples:
        dataset = dataset.select(range(num_samples))
    
    # Generate answers for each question
    all_predictions = []
    all_references = []
    
    for example in tqdm(dataset, desc="Evaluating QA"):
        # Get question and reference answer
        question = example.get('question', example.get('input', ""))
        reference = example.get('answer', example.get('output', ""))
        
        # Generate answer
        prediction = generate_text(
            question,
            max_length=100,
            temperature=0.3
        )[0]
        
        # Remove question from prediction
        prediction = prediction.replace(question, "").strip()
        
        # Add to lists
        all_predictions.append(prediction)
        all_references.append(reference)
    
    # Compute exact match
    exact_match = compute_exact_match(all_predictions, all_references)
    results["exact_match"] = exact_match
    
    # Compute F1 score
    f1_score = compute_f1(all_predictions, all_references)
    results["f1"] = f1_score
    
    return results

def evaluate_reasoning(dataset, num_samples=10):
    """Evaluate model on reasoning tasks."""
    results = {}
    
    # Limit dataset size for demonstration
    if len(dataset) > num_samples:
        dataset = dataset.select(range(num_samples))
    
    # Generate reasoning for each problem
    all_predictions = []
    all_references = []
    
    for example in tqdm(dataset, desc="Evaluating reasoning"):
        # Get problem and reference solution
        problem = example.get('problem', example.get('question', example.get('input', "")))
        reference = example.get('solution', example.get('answer', example.get('output', "")))
        
        # Generate solution
        prediction = generate_text(
            problem,
            max_length=200,
            temperature=0.3
        )[0]
        
        # Remove problem from prediction
        prediction = prediction.replace(problem, "").strip()
        
        # Add to lists
        all_predictions.append(prediction)
        all_references.append(reference)
    
    # For demonstration, use F1 score as a proxy for reasoning quality
    f1_score = compute_f1(all_predictions, all_references)
    results["reasoning_quality"] = f1_score
    
    return results

## Evaluate on Code Generation Benchmarks

Evaluate the model on code generation benchmarks such as HumanEval.

In [ ]:
# Load HumanEval dataset
try:
    humaneval_dataset = load_dataset("openai/humaneval", split="test")
    print(f"Loaded HumanEval dataset with {len(humaneval_dataset)} examples")
    
    # Evaluate on HumanEval
    humaneval_results = evaluate_code_generation(humaneval_dataset, num_samples=5)
    
    print("HumanEval results:")
    for metric, value in humaneval_results.items():
        print(f"  {metric}: {value:.4f}")
except Exception as e:
    print(f"Error loading or evaluating HumanEval dataset: {e}")
    humaneval_results = {"pass@1": 0.0, "pass@5": 0.0, "pass@10": 0.0}

## Evaluate on Dialogue and QA Benchmarks

Evaluate the model on dialogue and question answering benchmarks.

In [ ]:
# Load TruthfulQA dataset
try:
    truthfulqa_dataset = load_dataset("truthful_qa", "multiple_choice", split="validation")
    print(f"Loaded TruthfulQA dataset with {len(truthfulqa_dataset)} examples")
    
    # Prepare dataset for evaluation
    qa_examples = []
    for example in truthfulqa_dataset:
        qa_examples.append({
            "question": example["question"],
            "answer": example["mc1_targets"][example["mc1_targets.correct"].index(1)]
        })
    
    qa_dataset = Dataset.from_list(qa_examples)
    
    # Evaluate on TruthfulQA
    truthfulqa_results = evaluate_qa(qa_dataset, num_samples=5)
    
    print("TruthfulQA results:")
    for metric, value in truthfulqa_results.items():
        print(f"  {metric}: {value:.4f}")
except Exception as e:
    print(f"Error loading or evaluating TruthfulQA dataset: {e}")
    truthfulqa_results = {"exact_match": 0.0, "f1": 0.0}

In [ ]:
# Load dialogue dataset
try:
    dialogue_dataset = load_dataset("Anthropic/hh-rlhf", "harmless-base", split="train[:100]")
    print(f"Loaded dialogue dataset with {len(dialogue_dataset)} examples")
    
    # Prepare dataset for evaluation
    dialogue_examples = []
    for example in dialogue_dataset:
        # Split into prompt and response
        parts = example["chosen"].split("\nAssistant: ")
        if len(parts) >= 2:
            dialogue_examples.append({
                "prompt": parts[0] + "\nAssistant: ",
                "completion": parts[1]
            })
    
    dialogue_eval_dataset = Dataset.from_list(dialogue_examples)
    
    # Evaluate on dialogue dataset
    dialogue_results = evaluate_dialogue(dialogue_eval_dataset, num_samples=5)
    
    print("Dialogue results:")
    for metric, value in dialogue_results.items():
        print(f"  {metric}: {value:.4f}")
except Exception as e:
    print(f"Error loading or evaluating dialogue dataset: {e}")
    dialogue_results = {"bleu_1": 0.0, "rouge_1_f": 0.0, "meteor": 0.0}

## Evaluate on Reasoning Benchmarks

Evaluate the model on reasoning benchmarks such as GSM8K.

In [ ]:
# Load GSM8K dataset
try:
    gsm8k_dataset = load_dataset("gsm8k", "main", split="test")
    print(f"Loaded GSM8K dataset with {len(gsm8k_dataset)} examples")
    
    # Prepare dataset for evaluation
    reasoning_examples = []
    for example in gsm8k_dataset:
        reasoning_examples.append({
            "problem": example["question"],
            "solution": example["answer"]
        })
    
    reasoning_dataset = Dataset.from_list(reasoning_examples)
    
    # Evaluate on GSM8K
    reasoning_results = evaluate_reasoning(reasoning_dataset, num_samples=5)
    
    print("GSM8K results:")
    for metric, value in reasoning_results.items():
        print(f"  {metric}: {value:.4f}")
except Exception as e:
    print(f"Error loading or evaluating GSM8K dataset: {e}")
    reasoning_results = {"reasoning_quality": 0.0}

## Analyze Model Performance

Analyze the model's performance across different benchmarks.

In [ ]:
# Combine all results
all_results = {
    "Code Generation": humaneval_results,
    "Question Answering": truthfulqa_results,
    "Dialogue": dialogue_results,
    "Reasoning": reasoning_results
}

# Create a summary table
summary_data = []

for category, results in all_results.items():
    for metric, value in results.items():
        summary_data.append({
            "Category": category,
            "Metric": metric,
            "Value": value
        })

summary_df = pd.DataFrame(summary_data)

# Display summary table
print("Model Performance Summary:")
print(summary_df)

# Create a bar chart
plt.figure(figsize=(12, 8))
chart = sns.barplot(x="Metric", y="Value", hue="Category", data=summary_df)
chart.set_xticklabels(chart.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title("Model Performance Across Benchmarks")
plt.tight_layout()
plt.show()

## Generate Evaluation Report

Generate a comprehensive evaluation report.

In [ ]:
# Create output directory
output_dir = "../outputs/evaluation"
os.makedirs(output_dir, exist_ok=True)

# Save results as JSON
with open(os.path.join(output_dir, "evaluation_results.json"), 'w') as f:
    json.dump(all_results, f, indent=2)

# Save summary table as CSV
summary_df.to_csv(os.path.join(output_dir, "evaluation_summary.csv"), index=False)

# Save chart as PNG
plt.figure(figsize=(12, 8))
chart = sns.barplot(x="Metric", y="Value", hue="Category", data=summary_df)
chart.set_xticklabels(chart.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title("Model Performance Across Benchmarks")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "evaluation_chart.png"), dpi=300)

# Generate HTML report
html_report = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Model Evaluation Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1 {{ color: #2c3e50; }}
        h2 {{ color: #3498db; }}
        table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f2f2f2; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .chart {{ margin: 20px 0; max-width: 100%; }}
    </style>
</head>
<body>
    <h1>Model Evaluation Report</h1>
    
    <h2>Summary</h2>
    <table>
        <tr>
            <th>Category</th>
            <th>Metric</th>
            <th>Value</th>
        </tr>
        {summary_df.to_html(index=False, header=False, classes='summary-table')}
    </table>
    
    <h2>Performance Chart</h2>
    <img src="evaluation_chart.png" alt="Performance Chart" class="chart">
    
    <h2>Detailed Results</h2>
"""

# Add detailed results for each category
for category, results in all_results.items():
    html_report += f"""
    <h3>{category}</h3>
    <table>
        <tr>
            <th>Metric</th>
            <th>Value</th>
        </tr>
    """
    
    for metric, value in results.items():
        html_report += f"""
        <tr>
            <td>{metric}</td>
            <td>{value:.4f}</td>
        </tr>
        """
    
    html_report += "</table>\n"

html_report += """
</body>
</html>
"""

# Save HTML report
with open(os.path.join(output_dir, "evaluation_report.html"), 'w') as f:
    f.write(html_report)

print(f"Evaluation report saved to {output_dir}")

## Summary

In this notebook, we have:

1. Loaded the trained model
2. Evaluated the model on code generation benchmarks
3. Evaluated the model on dialogue and QA benchmarks
4. Evaluated the model on reasoning benchmarks
5. Analyzed the model's performance across different tasks
6. Generated a comprehensive evaluation report

The evaluation results provide insights into the model's strengths and weaknesses across different tasks, which can guide further improvements and optimizations.